<a href="https://colab.research.google.com/github/awa-easymoneysniper/OSU-Mania-Auto-Beatmap-Generator/blob/main/mania_event_encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path
from bisect import bisect_right
from collections import defaultdict, Counter

import torch
from tqdm.auto import tqdm

In [ ]:
# =========================
# 1. Event 定义
# =========================

@dataclass
class Event:
    type: str          # "tap" | "hold_start" | "hold_end"
    beat: float        # 以第一个红线 timing point 为 beat=0 的绝对 beat
    lane: int          # 4K 时范围为 0, 1, 2, 3

In [ ]:
# =========================
# 2. osu!mania 谱面解析器
# =========================

class BeatmapParser:
    def __init__(self, target_keys=4):
        self.target_keys = target_keys

    def _timing_points(self, osu_text):
        """
        读取所有非继承 timing point（红线）。
        返回 [(time_ms, beat_length_ms), ...]
        """
        timing_points = []
        in_timing = False

        for raw_line in osu_text.splitlines():
            line = raw_line.strip()

            if line == "[TimingPoints]":
                in_timing = True
                continue

            if not in_timing:
                continue

            if line.startswith("["):
                break

            if not line:
                continue

            s = line.split(",")

            # timing point 格式至少应有 7 项
            if len(s) < 7:
                continue

            # s[6] == 1 表示非继承 timing point，即真正 BPM 红线
            if int(s[6]) == 1:
                time_ms = float(s[0])
                beat_length_ms = float(s[1])

                if beat_length_ms > 0:
                    timing_points.append((time_ms, beat_length_ms))

        if not timing_points:
            raise RuntimeError("No uninherited timing point found.")

        timing_points.sort(key=lambda x: x[0])
        return timing_points

    def _build_ms_to_beat(self, timing_points):
        """
        建立 time_ms -> absolute beat 的转换函数。

        beat=0 定义为第一个非继承 timing point。
        后续如果发生 BPM 变化，会正确累计 beat 数。
        """
        times = [x[0] for x in timing_points]
        beat_lengths = [x[1] for x in timing_points]

        cumulative_beats = [0.0]

        for i in range(1, len(times)):
            elapsed_ms = times[i] - times[i - 1]
            beats = elapsed_ms / beat_lengths[i - 1]
            cumulative_beats.append(cumulative_beats[-1] + beats)

        def ms_to_beat(time_ms):
            # 找到不大于 time_ms 的最后一个 timing point
            idx = bisect_right(times, time_ms) - 1

            # 少数谱面的第一个 note 可能比第一条 timing point 早 1ms
            if idx < 0:
                return (time_ms - times[0]) / beat_lengths[0]

            return (
                cumulative_beats[idx]
                + (time_ms - times[idx]) / beat_lengths[idx]
            )

        return ms_to_beat

    def parse(self, beatmap):
        """
        输入单张 beatmap dict。
        输出 events 和 metadata。
        """
        osu_text = beatmap["osu"]
        keys = int(beatmap["keys"])

        # 强制只处理 4K
        if keys != self.target_keys:
            raise ValueError(
                f"Only {self.target_keys}K beatmaps are supported, got {keys}K."
            )

        timing_points = self._timing_points(osu_text)
        ms_to_beat = self._build_ms_to_beat(timing_points)

        events = []
        in_hitobjects = False

        for raw_line in osu_text.splitlines():
            line = raw_line.strip()

            if line == "[HitObjects]":
                in_hitobjects = True
                continue

            if not in_hitobjects:
                continue

            # HitObjects 后若还有 section，停止解析
            if line.startswith("["):
                break

            if not line:
                continue

            s = line.split(",")

            if len(s) < 5:
                continue

            x = int(s[0])
            start_ms = int(s[2])
            obj_type = int(s[3])

            # mania 坐标 x 通常范围是 [0, 512)
            lane = int(x * keys / 512)
            lane = max(0, min(keys - 1, lane))

            start_beat = ms_to_beat(start_ms)

            # 长按：type 的第 7 位为 1，即 obj_type & 128
            if obj_type & 128:
                if len(s) < 6:
                    continue

                end_ms = int(s[5].split(":")[0])
                end_beat = ms_to_beat(end_ms)

                events.append(Event(
                    type="hold_start",
                    beat=start_beat,
                    lane=lane
                ))

                events.append(Event(
                    type="hold_end",
                    beat=end_beat,
                    lane=lane
                ))

            # 普通 note，包括 obj_type=1、5 等
            else:
                events.append(Event(
                    type="tap",
                    beat=start_beat,
                    lane=lane
                ))

        # 同一拍内先处理 hold_end，再处理 hold_start / tap
        # 对之后可能的 lane 状态建模更安全
        event_priority = {
            "hold_end": 0,
            "hold_start": 1,
            "tap": 2,
        }

        events.sort(key=lambda e: (e.beat, event_priority[e.type], e.lane))

        return {
            "events": events,
            "metadata": {
                "beatmap_id": str(beatmap["beatmap_id"]),
                "beatmapset_id": str(beatmap["beatmapset_id"]),
                "difficulty_name": beatmap["difficulty_name"],
                "star_rating": float(beatmap["star_rating"]),
                "keys": keys,
                "bpm": float(beatmap["bpm"]),
                "length": int(beatmap["length"]),
                "notes": int(beatmap["notes"]),
                "long_notes": int(beatmap["long_notes"]),
            }
        }

In [ ]:
# =========================
# 3. Event tokenizer
# =========================

class EventTokenizer:
    EVENT2ID = {
        "tap": 0,
        "hold_start": 1,
        "hold_end": 2,
    }

    def __init__(self, beat_resolution=48, max_delta=768):
        """
        beat_resolution=48:
        1/4 beat  -> 12
        1/8 beat  -> 6
        1/16 beat -> 3

        max_delta=768 表示最多保留 16 beat 的空白间隔。
        过长间隔截断，防止 delta token 的词表无限增大。
        """
        self.resolution = beat_resolution
        self.max_delta = max_delta

    def encode(self, parsed):
        events = parsed["events"]

        previous_beat = 0.0
        sequence = []

        for event in events:
            delta_beat = event.beat - previous_beat
            previous_beat = event.beat

            delta_token = round(delta_beat * self.resolution)
            delta_token = max(0, min(delta_token, self.max_delta))

            # 每一行: [delta_beat_token, event_type_id, lane]
            sequence.append([
                delta_token,
                self.EVENT2ID[event.type],
                event.lane,
            ])

        return {
            "events": torch.tensor(sequence, dtype=torch.long),

            # 保留原始连续元数据；归一化应在训练集划分后进行
            "metadata": torch.tensor([
                parsed["metadata"]["star_rating"],
                parsed["metadata"]["keys"],
                parsed["metadata"]["bpm"],
                parsed["metadata"]["length"],
                parsed["metadata"]["notes"],
                parsed["metadata"]["long_notes"],
            ], dtype=torch.float32)
        }

In [ ]:
# =========================
# 4. 批量处理全部 JSON，并分片保存
# =========================

INPUT_DIR = Path("/content/drive/MyDrive/osu_mania_dataset/metadata")

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/osu_mania_dataset/beatmap_events_4k"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = OUTPUT_DIR / "beatmap_index.json"

TARGET_KEYS = 4
SHARD_SIZE = 512

parser = BeatmapParser(target_keys=TARGET_KEYS)
tokenizer = EventTokenizer(
    beat_resolution=48,
    max_delta=768,
)

# 当前尚未写入磁盘的谱面
pending_samples = []

# 全局索引：audio_id -> 多张谱面的位置
audio_to_samples = defaultdict(list)

# 全局索引：beatmap_id -> 单张谱面的位置
beatmap_id_to_sample = {}

errors = []
stats = Counter()

current_shard_id = 0


def flush_shard():
    """将当前 pending_samples 写成一个 shard 文件。"""
    global current_shard_id, pending_samples

    if not pending_samples:
        return

    shard_path = OUTPUT_DIR / f"shard_{current_shard_id:05d}.pt"

    torch.save(
        {
            "shard_id": current_shard_id,
            "samples": pending_samples,
        },
        shard_path,
    )

    print(
        f"Saved {shard_path.name}: "
        f"{len(pending_samples)} beatmaps"
    )

    pending_samples = []
    current_shard_id += 1


json_files = sorted(INPUT_DIR.rglob("*.json"))

for json_path in tqdm(json_files, desc="Processing JSON files"):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            song_data = json.load(f)

        # 例如 "008798.mp3" -> "008798"
        # 保持字符串类型，不能转为 int
        audio_file = song_data["audio_file"]
        audio_id = Path(audio_file).stem

        stats["audio_files"] += 1

        for beatmap in song_data.get("beatmaps", []):
            try:
                # 仅保留 4K
                if int(beatmap["keys"]) != TARGET_KEYS:
                    stats["skipped_non_4k"] += 1
                    continue

                beatmap_id = str(beatmap["beatmap_id"])

                if beatmap_id in beatmap_id_to_sample:
                    raise ValueError(f"Duplicate beatmap_id: {beatmap_id}")

                parsed = parser.parse(beatmap)
                encoded = tokenizer.encode(parsed)

                if encoded["events"].shape[0] == 0:
                    stats["skipped_empty"] += 1
                    continue

                # 这张谱面在当前 shard 内的位置
                offset = len(pending_samples)

                # 保存位置，后续可通过它精确读取谱面
                location = {
                    "shard": current_shard_id,
                    "offset": offset,
                    "beatmap_id": beatmap_id,
                }

                sample = {
                    # 音频关联信息
                    "audio_id": audio_id,
                    "audio_file": audio_file,

                    # 谱面标识
                    "beatmap_id": beatmap_id,
                    "pair_id": f"{audio_id}:{beatmap_id}",
                    "beatmapset_id": parsed["metadata"]["beatmapset_id"],

                    # 难度与统计信息
                    "difficulty_name": parsed["metadata"]["difficulty_name"],
                    "star_rating": parsed["metadata"]["star_rating"],
                    "keys": parsed["metadata"]["keys"],
                    "bpm": parsed["metadata"]["bpm"],
                    "length": parsed["metadata"]["length"],
                    "notes": parsed["metadata"]["notes"],
                    "long_notes": parsed["metadata"]["long_notes"],

                    # [num_events, 3]
                    # 列: [delta_beat_token, event_type_id, lane]
                    #
                    # int16 足够：
                    # delta <= 768, type <= 2, lane <= 3
                    "events": encoded["events"].to(torch.int16),

                    # [star_rating, keys, bpm, length, notes, long_notes]
                    "metadata": encoded["metadata"],

                    "source_json": json_path.name,
                }

                pending_samples.append(sample)

                # 同一音频可对应多张不同难度谱面
                audio_to_samples[audio_id].append(location)
                beatmap_id_to_sample[beatmap_id] = location

                stats["processed_4k"] += 1

                # 当前 shard 满时写入磁盘
                if len(pending_samples) >= SHARD_SIZE:
                    flush_shard()

            except Exception as e:
                stats["beatmap_errors"] += 1
                errors.append({
                    "source_json": json_path.name,
                    "beatmap_id": beatmap.get("beatmap_id"),
                    "error": repr(e),
                })

    except Exception as e:
        stats["json_errors"] += 1
        errors.append({
            "source_json": json_path.name,
            "beatmap_id": None,
            "error": repr(e),
        })

# 保存最后不足 SHARD_SIZE 的部分
flush_shard()


# =========================
# 5. 保存全局索引
# =========================

beatmap_index = {
    "config": {
        "target_keys": TARGET_KEYS,
        "shard_size": SHARD_SIZE,
        "beat_resolution": tokenizer.resolution,
        "max_delta": tokenizer.max_delta,
        "event2id": tokenizer.EVENT2ID,
        "event_columns": [
            "delta_beat_token",
            "event_type_id",
            "lane",
        ],
        "metadata_order": [
            "star_rating",
            "keys",
            "bpm",
            "length",
            "notes",
            "long_notes",
        ],
    },

    # audio_id -> 多张谱面的位置
    "audio_to_samples": dict(audio_to_samples),

    # beatmap_id -> 单张谱面的位置
    "beatmap_id_to_sample": beatmap_id_to_sample,

    "stats": dict(stats),
    "errors": errors,
}

with open(INDEX_PATH, "w", encoding="utf-8") as f:
    json.dump(
        beatmap_index,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\n========== Finished ==========")
print(f"JSON 文件数: {len(json_files)}")
print(f"音频数量: {len(audio_to_samples)}")
print(f"处理完成的 4K 谱面数: {stats['processed_4k']}")
print(f"跳过的非 4K 谱面数: {stats['skipped_non_4k']}")
print(f"生成 shard 数量: {current_shard_id}")
print(f"索引文件: {INDEX_PATH}")

Processing JSON files:   0%|          | 0/8799 [00:00<?, ?it/s]

Saved shard_00000.pt: 512 beatmaps
Saved shard_00001.pt: 512 beatmaps
Saved shard_00002.pt: 512 beatmaps
Saved shard_00003.pt: 512 beatmaps
Saved shard_00004.pt: 512 beatmaps
Saved shard_00005.pt: 512 beatmaps
Saved shard_00006.pt: 512 beatmaps
Saved shard_00007.pt: 512 beatmaps
Saved shard_00008.pt: 512 beatmaps
Saved shard_00009.pt: 512 beatmaps
Saved shard_00010.pt: 512 beatmaps
Saved shard_00011.pt: 512 beatmaps
Saved shard_00012.pt: 512 beatmaps
Saved shard_00013.pt: 512 beatmaps
Saved shard_00014.pt: 512 beatmaps
Saved shard_00015.pt: 512 beatmaps
Saved shard_00016.pt: 512 beatmaps
Saved shard_00017.pt: 512 beatmaps
Saved shard_00018.pt: 512 beatmaps
Saved shard_00019.pt: 512 beatmaps
Saved shard_00020.pt: 512 beatmaps
Saved shard_00021.pt: 512 beatmaps
Saved shard_00022.pt: 512 beatmaps
Saved shard_00023.pt: 512 beatmaps
Saved shard_00024.pt: 512 beatmaps
Saved shard_00025.pt: 512 beatmaps
Saved shard_00026.pt: 512 beatmaps
Saved shard_00027.pt: 512 beatmaps
Saved shard_00028.pt

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
from tqdm.auto import tqdm

INPUT_DIR = Path("/content/drive/MyDrive/osu_mania_dataset/metadata")

map_key_counts = Counter()
file_category_counts = Counter()

audio_to_jsons = defaultdict(list)
files_without_4k = []
files_with_4k = []
json_read_errors = []

for json_path in tqdm(sorted(INPUT_DIR.rglob("*.json")), desc="Diagnosing"):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            song_data = json.load(f)

        audio_id = Path(song_data["audio_file"]).stem
        audio_to_jsons[audio_id].append(json_path.name)

        beatmaps = song_data.get("beatmaps", [])
        key_counts_in_file = Counter(int(x["keys"]) for x in beatmaps)

        for keys, count in key_counts_in_file.items():
            map_key_counts[keys] += count

        if key_counts_in_file[4] > 0:
            file_category_counts["has_4k"] += 1
            files_with_4k.append(json_path.name)
        else:
            file_category_counts["no_4k"] += 1
            files_without_4k.append({
                "json": json_path.name,
                "audio_file": song_data["audio_file"],
                "keys_in_file": dict(key_counts_in_file),
            })

    except Exception as e:
        json_read_errors.append({
            "json": json_path.name,
            "error": repr(e),
        })

duplicate_audio_ids = {
    audio_id: json_list
    for audio_id, json_list in audio_to_jsons.items()
    if len(json_list) > 1
}

print("========== Key 数量分布 ==========")
for keys in sorted(map_key_counts):
    print(f"{keys}K 谱面数: {map_key_counts[keys]}")

print("\n========== JSON 文件情况 ==========")
print(f"总 JSON 文件数: {sum(file_category_counts.values()) + len(json_read_errors)}")
print(f"含至少一张 4K 谱面的 JSON: {file_category_counts['has_4k']}")
print(f"完全没有 4K 谱面的 JSON: {file_category_counts['no_4k']}")
print(f"JSON 读取失败数: {len(json_read_errors)}")

print("\n========== audio_id 去重情况 ==========")
print(f"唯一 audio_id 数: {len(audio_to_jsons)}")
print(f"被多个 JSON 共用的 audio_id 数: {len(duplicate_audio_ids)}")

print("\n========== 前 10 个无 4K 的例子 ==========")
for x in files_without_4k[:10]:
    print(x)

print("\n========== 前 10 个 audio_id 重复例子 ==========")
for audio_id, json_list in list(duplicate_audio_ids.items())[:10]:
    print(audio_id, "->", json_list)

Diagnosing:   0%|          | 0/8799 [00:00<?, ?it/s]

========== Key 数量分布 ==========
1K 谱面数: 22
2K 谱面数: 5
4K 谱面数: 19408
5K 谱面数: 679
6K 谱面数: 920
7K 谱面数: 5016
8K 谱面数: 289
9K 谱面数: 202
10K 谱面数: 502
12K 谱面数: 48
14K 谱面数: 9
16K 谱面数: 48
18K 谱面数: 32

========== JSON 文件情况 ==========
总 JSON 文件数: 8799
含至少一张 4K 谱面的 JSON: 6998
完全没有 4K 谱面的 JSON: 1801
JSON 读取失败数: 0

========== audio_id 去重情况 ==========
唯一 audio_id 数: 8799
被多个 JSON 共用的 audio_id 数: 0

========== 前 10 个无 4K 的例子 ==========
{'json': '000002.json', 'audio_file': '000002.mp3', 'keys_in_file': {7: 2}}
{'json': '000020.json', 'audio_file': '000020.mp3', 'keys_in_file': {7: 5}}
{'json': '000038.json', 'audio_file': '000038.mp3', 'keys_in_file': {7: 3}}
{'json': '000039.json', 'audio_file': '000039.mp3', 'keys_in_file': {12: 2}}
{'json': '000040.json', 'audio_file': '000040.mp3', 'keys_in_file': {7: 1}}
{'json': '000042.json', 'audio_file': '000042.mp3', 'keys_in_file': {7: 1}}
{'json': '000043.json', 'audio_file': '000043.mp3', 'keys_in_file': {8: 3, 5: 3}}
{'json': '000045.json', 'audio_file': '00